In [1]:
import pandas as pd

fund_master = pd.read_csv("../data/raw/01_fund_master.csv")
nav_history = pd.read_csv("../data/raw/02_nav_history.csv")
investor_transactions = pd.read_csv("../data/raw/08_investor_transactions.csv")
scheme_performance = pd.read_csv("../data/raw/07_scheme_performance.csv")

In [2]:
nav_history.head()

,amfi_code,date,nav
0,119551,2022-01-03,54.3856
1,119551,2022-01-04,54.3474
2,119551,2022-01-05,54.6869
3,119551,2022-01-06,55.4550
4,119551,2022-01-07,55.3692


In [4]:
nav_history['date'] = pd.to_datetime(nav_history['date'])

nav_history = nav_history.sort_values(
    ['amfi_code','date']
)

nav_history = nav_history.drop_duplicates()

nav_history = nav_history[nav_history['nav'] > 0]

In [5]:
nav_history.info()

<class 'pandas.DataFrame'>
Index: 46000 entries, 5750 to 45999
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   amfi_code  46000 non-null  int64         
 1   date       46000 non-null  datetime64[us]
 2   nav        46000 non-null  float64       
dtypes: datetime64[us](1), float64(1), int64(1)
memory usage: 1.4 MB


In [6]:
nav_history.to_csv(
    "../data/processed/nav_history_clean.csv",
    index=False
)

In [7]:
investor_transactions.head()

,investor_id,transaction_date,amfi_code,transaction_type,amount_inr,state,city,city_tier,age_group,gender,annual_income_lakh,payment_mode,kyc_status
0,INV003054,2024-01-01,119092,SIP,1834,Telangana,Hyderabad,T30,56+,Female,77.1,UPI,Verified
1,INV002952,2024-01-01,148567,Redemption,392882,Punjab,Amritsar,B30,18-25,Male,7.1,Cheque,Verified
2,INV003420,2024-01-01,118636,SIP,912,Haryana,Faridabad,B30,36-45,Male,47.2,Mandate,Verified
3,INV003436,2024-01-01,118634,SIP,1102,Maharashtra,Mumbai,T30,36-45,Female,54.4,Cheque,Pending
4,INV004691,2024-01-01,119094,Lumpsum,8682,Delhi,Noida,T30,26-35,Male,14.5,Net Banking,Pending


In [8]:
investor_transactions.columns

Index(['investor_id', 'transaction_date', 'amfi_code', 'transaction_type',
       'amount_inr', 'state', 'city', 'city_tier', 'age_group', 'gender',
       'annual_income_lakh', 'payment_mode', 'kyc_status'],
      dtype='str')

In [9]:
# Clean transaction date

investor_transactions['transaction_date'] = pd.to_datetime(
    investor_transactions['transaction_date']
)

# Amount should be positive

investor_transactions = investor_transactions[
    investor_transactions['amount_inr'] > 0
]

# Standardize transaction types

investor_transactions['transaction_type'] = (
    investor_transactions['transaction_type']
    .str.strip()
    .str.title()
)

# Check valid transaction types

print(
    investor_transactions['transaction_type']
    .value_counts()
)

transaction_type
Sip           19716
Lumpsum        8095
Redemption     4967
Name: count, dtype: int64


In [10]:
investor_transactions.to_csv(
    "../data/processed/investor_transactions_clean.csv",
    index=False
)

In [11]:
scheme_performance.columns

Index(['amfi_code', 'scheme_name', 'fund_house', 'category', 'plan',
       'return_1yr_pct', 'return_3yr_pct', 'return_5yr_pct',
       'benchmark_3yr_pct', 'alpha', 'beta', 'sharpe_ratio', 'sortino_ratio',
       'std_dev_ann_pct', 'max_drawdown_pct', 'aum_crore', 'expense_ratio_pct',
       'morningstar_rating', 'risk_grade'],
      dtype='str')

In [12]:
# Convert percentage columns to numeric

cols = [
    'return_1yr_pct',
    'return_3yr_pct',
    'return_5yr_pct',
    'benchmark_3yr_pct',
    'alpha',
    'beta',
    'sharpe_ratio',
    'sortino_ratio',
    'std_dev_ann_pct',
    'max_drawdown_pct',
    'aum_crore',
    'expense_ratio_pct'
]

for col in cols:
    scheme_performance[col] = pd.to_numeric(
        scheme_performance[col],
        errors='coerce'
    )

# Remove duplicate rows
scheme_performance = scheme_performance.drop_duplicates()

# Expense ratio should be between 0 and 2.5
scheme_performance = scheme_performance[
    (scheme_performance['expense_ratio_pct'] >= 0) &
    (scheme_performance['expense_ratio_pct'] <= 2.5)
]

scheme_performance.head()

,amfi_code,scheme_name,fund_house,category,plan,return_1yr_pct,return_3yr_pct,return_5yr_pct,benchmark_3yr_pct,alpha,beta,sharpe_ratio,sortino_ratio,std_dev_ann_pct,max_drawdown_pct,aum_crore,expense_ratio_pct,morningstar_rating,risk_grade
0,119551,SBI Bluechip Fund - Regular Plan - Growth,SBI Mutual Fund,Large Cap,Regular,12.42,12.36,14.45,11.49,0.87,0.89,0.88,1.29,14.0,-21.70,14288,1.54,4,Moderate
1,119552,SBI Bluechip Fund - Direct Plan - Growth,SBI Mutual Fund,Large Cap,Direct,15.25,11.30,14.23,9.52,1.78,0.87,0.81,1.29,14.0,-24.43,1231,0.66,3,Moderate
2,119598,SBI Small Cap Fund - Regular Plan - Growth,SBI Mutual Fund,Small Cap,Regular,24.56,23.39,20.67,22.16,1.23,0.89,0.94,1.35,25.0,-13.35,19259,1.43,5,Very High
3,119599,SBI Small Cap Fund - Direct Plan - Growth,SBI Mutual Fund,Small Cap,Direct,20.59,23.14,21.82,22.01,1.13,1.04,0.93,1.67,25.0,-24.78,36061,0.72,4,Very High
4,119120,SBI Magnum Gilt Fund - Regular Plan - Growth,SBI Mutual Fund,Gilt,Regular,5.34,6.07,5.43,4.47,1.60,0.22,1.52,2.11,4.0,-2.30,24101,0.77,5,Low


In [13]:
scheme_performance.isnull().sum()

amfi_code             0
scheme_name           0
fund_house            0
category              0
plan                  0
return_1yr_pct        0
return_3yr_pct        0
return_5yr_pct        0
benchmark_3yr_pct     0
alpha                 0
beta                  0
sharpe_ratio          0
sortino_ratio         0
std_dev_ann_pct       0
max_drawdown_pct      0
aum_crore             0
expense_ratio_pct     0
morningstar_rating    0
risk_grade            0
dtype: int64

In [14]:
scheme_performance.to_csv(
    "../data/processed/scheme_performance_clean.csv",
    index=False
)

In [15]:
import os

raw_path = "../data/raw"
processed_path = "../data/processed"

for file in os.listdir(raw_path):
    
    if file.endswith(".csv"):
        
        df = pd.read_csv(f"{raw_path}/{file}")
        
        df = df.drop_duplicates()
        
        output_name = file.replace(".csv","_clean.csv")
        
        df.to_csv(
            f"{processed_path}/{output_name}",
            index=False
        )

print("All datasets cleaned and saved")

All datasets cleaned and saved


In [16]:
import sqlite3

conn = sqlite3.connect("../data/db/bluestock_mf.db")

print("Database Created Successfully")

Database Created Successfully


In [17]:
import pandas as pd
import sqlite3
import os

conn = sqlite3.connect("../data/db/bluestock_mf.db")

processed_path = "../data/processed"

for file in os.listdir(processed_path):

    if file.endswith(".csv"):

        table_name = file.replace(".csv","")

        df = pd.read_csv(f"{processed_path}/{file}")

        df.to_sql(
            table_name,
            conn,
            if_exists="replace",
            index=False
        )

        print(f"{table_name} loaded")

conn.close()

print("All tables loaded into SQLite")

01_fund_master_clean loaded
02_nav_history_clean loaded
03_aum_by_fund_house_clean loaded
04_monthly_sip_inflows_clean loaded
05_category_inflows_clean loaded
06_industry_folio_count_clean loaded
07_scheme_performance_clean loaded
08_investor_transactions_clean loaded
09_portfolio_holdings_clean loaded
10_benchmark_indices_clean loaded
118632_live_nav_clean loaded
119092_live_nav_clean loaded
119551_live_nav_clean loaded
120503_live_nav_clean loaded
120841_live_nav_clean loaded
hdfc_live_nav_clean loaded
investor_transactions_clean loaded
nav_history_clean loaded
scheme_performance_clean loaded
All tables loaded into SQLite


In [18]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("../data/db/bluestock_mf.db")

query = """
SELECT name
FROM sqlite_master
WHERE type='table';
"""

pd.read_sql(query, conn)

,name
0,01_fund_master_clean
1,02_nav_history_clean
2,03_aum_by_fund_house_clean
3,04_monthly_sip_inflows_clean
4,05_category_inflows_clean
5,06_industry_folio_count_clean
6,07_scheme_performance_clean
7,08_investor_transactions_clean
8,09_portfolio_holdings_clean
9,10_benchmark_indices_clean
